# 🖼️ Image Lab — Streamlit 版（在 Colab 上執行）

**使用方式：** 「執行階段 → 全部執行」，等最後一格印出 **公開網址**，點開即可使用 Streamlit 介面。

| 步驟 | 說明 |
|---|---|
| 1 | 安裝 Streamlit、檢查 OpenCV 版本 |
| 2 | 把 `app.py` 寫入 Colab 工作目錄 |
| 3 | 下載 YuNet 人臉模型（約 230 KB） |
| 4 | 啟動 Streamlit，並用 cloudflared 產生臨時公開網址 |

> ⚠️ 網址只在此 Colab 執行階段運作中有效；關閉分頁或閒置過久，網址就會失效，重新執行最後一格即可。


## 1. 安裝與環境檢查

In [1]:
import subprocess, sys
import cv2

# Streamlit（需 >= 1.50）。Colab 已內建 OpenCV / NumPy / Matplotlib / Pandas / Pillow，不另外安裝以免衝突。
try:
    import streamlit
    _ok = tuple(int(x) for x in streamlit.__version__.split(".")[:2]) >= (1, 50)
except ImportError:
    _ok = False
if not _ok:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "streamlit>=1.50,<2"])

major, minor = (int(x) for x in cv2.__version__.split(".")[:2])
if (major, minor) < (4, 8):  # YuNet 需要 OpenCV >= 4.8
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless>=4.10"])
    print("已升級 OpenCV，請「執行階段 → 重新啟動工作階段」後再從頭執行。")

import streamlit, numpy, PIL, matplotlib, pandas
print("Python   :", sys.version.split()[0])
print("OpenCV   :", cv2.__version__)
for m in (streamlit, numpy, PIL, matplotlib, pandas):
    print(f"{m.__name__:9s}:", m.__version__)


Python   : 3.13.18
OpenCV   : 4.14.0
streamlit: 1.66.1
numpy    : 2.1.3
PIL      : 11.3.0
matplotlib: 3.10.0
pandas   : 2.2.3


## 2. 寫入 `app.py`（Streamlit 程式本體）

In [2]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""
Image Lab — 影像處理期中專題
=================================================
功能
  1. 影像載入與基礎解析 (PNG / JPG / JPEG / BMP + 中繼資訊)
  2. 多色彩空間檢視 (Original / R,G,B / Y,U,V) 與同步直方圖
  3. 自訂卷積核空間濾波 + PSNR 量化評估
  4. 人臉偵測 (YuNet / Haar Cascade)，輸出邊界框、人臉數、信心度

執行：  streamlit run app.py
"""
from __future__ import annotations

import io
import math
import os
import re
import tempfile
import urllib.request
from fractions import Fraction

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import streamlit as st  # noqa: E402
from PIL import Image, ImageOps  # noqa: E402

# ----------------------------------------------------------------------------
# 常數
# ----------------------------------------------------------------------------
ALLOWED_EXT = ["png", "jpg", "jpeg", "bmp"]
MAX_KERNEL = 15  # 卷積核最大邊長

YUNET_NAME = "face_detection_yunet_2023mar.onnx"
# 注意：raw.githubusercontent.com 對此檔只會回傳 Git LFS 指標檔(約 131 bytes)，
# 因此優先使用 github.com/.../raw/ (會轉址至 LFS 實體檔)；下載後以檔案大小驗證。
_YUNET_PATH = "opencv/opencv_zoo/main/models/face_detection_yunet/" + YUNET_NAME
YUNET_URLS = [
    "https://github.com/" + _YUNET_PATH.replace("/main/", "/raw/main/", 1),
    "https://media.githubusercontent.com/media/" + _YUNET_PATH,
]

# YUV (BT.601, analog) 理論值域：RGB ∈ [0,255] 時
U_MAX = 0.436 * 255.0  # ≈ 111.18，U ∈ [-U_MAX, +U_MAX]
V_MAX = 0.615 * 255.0  # ≈ 156.83，V ∈ [-V_MAX, +V_MAX]

# 邊界處理：(cv2 borderType, np.pad mode)
BORDERS = {
    "Reflect（鏡射）": (cv2.BORDER_REFLECT_101, "reflect"),
    "Replicate（複製邊緣）": (cv2.BORDER_REPLICATE, "edge"),
    "Zero padding（補 0）": (cv2.BORDER_CONSTANT, "constant"),
}

PRESETS = {
    "Identity（單位）": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], float),
    "邊緣銳化 Sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    "高斯平滑 3×3": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], float),
    "高斯平滑 5×5": np.array(
        [
            [1, 4, 6, 4, 1],
            [4, 16, 24, 16, 4],
            [6, 24, 36, 24, 6],
            [4, 16, 24, 16, 4],
            [1, 4, 6, 4, 1],
        ],
        float,
    ),
    "均值模糊 3×3": np.ones((3, 3), float),
    "浮雕 Emboss": np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], float),
    "Sobel X（邊緣）": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Laplacian（邊緣）": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
    "自訂 Custom": None,
}


# ----------------------------------------------------------------------------
# 1. 影像載入
# ----------------------------------------------------------------------------
def decode_image(data: bytes, max_side: int = 1600):
    """bytes → (RGB uint8 ndarray, meta dict)。處理 alpha、EXIF 旋轉、16-bit、過大影像。"""
    img = Image.open(io.BytesIO(data))
    img.load()
    fmt, mode, size = img.format, img.mode, img.size
    channels = len(img.getbands())

    if mode in ("I;16", "I;16L", "I;16B", "I", "F"):
        bit_depth = 16 if mode.startswith("I") else 32
        arr = np.asarray(img, dtype=np.float64)
        peak = arr.max()
        arr = arr / peak * 255.0 if peak > 0 else arr
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).convert("RGB")
    else:
        bit_depth = 1 if mode == "1" else 8
        img = ImageOps.exif_transpose(img)
        has_alpha = img.mode in ("RGBA", "LA", "PA") or (
            img.mode == "P" and "transparency" in img.info
        )
        if has_alpha:  # 透明背景 → 墊白底，避免全黑
            rgba = img.convert("RGBA")
            bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            img = Image.alpha_composite(bg, rgba).convert("RGB")
        else:
            img = img.convert("RGB")

    rgb = np.array(img, dtype=np.uint8)
    meta = {
        "format": fmt or "未知",
        "mode": mode,
        "channels": channels,
        "bit_depth": bit_depth,
        "orig_size": size,  # (W, H)
        "file_bytes": len(data),
        "resized": False,
    }

    h, w = rgb.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        rgb = cv2.resize(
            rgb, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA
        )
        meta["resized"] = True
    meta["proc_size"] = (rgb.shape[1], rgb.shape[0])
    return rgb, meta


@st.cache_data(show_spinner=False)
def load_image(data: bytes, max_side: int):
    return decode_image(data, max_side)


def human_size(n: int) -> str:
    if n < 1024:
        return f"{n} B"
    if n < 1024**2:
        return f"{n / 1024:.1f} KB"
    return f"{n / 1024**2:.2f} MB"


# ----------------------------------------------------------------------------
# 2. 色彩空間與直方圖
# ----------------------------------------------------------------------------
def rgb_to_yuv(rgb: np.ndarray):
    """BT.601：回傳浮點 (Y, U, V)；Y∈[0,255]，U/V 含負值。"""
    f = rgb.astype(np.float64)
    R, G, B = f[..., 0], f[..., 1], f[..., 2]
    Y = 0.299 * R + 0.587 * G + 0.114 * B
    U = -0.14713 * R - 0.28886 * G + 0.436 * B
    V = 0.615 * R - 0.51499 * G - 0.10001 * B
    return Y, U, V


def yuv_channel_u8(rgb: np.ndarray, ch: str) -> np.ndarray:
    """Y 直接取用；U/V 以理論值域線性映射到 0–255 後輸出 uint8。"""
    Y, U, V = rgb_to_yuv(rgb)
    if ch == "Y":
        out = Y
    elif ch == "U":
        out = (U + U_MAX) / (2 * U_MAX) * 255.0
    else:
        out = (V + V_MAX) / (2 * V_MAX) * 255.0
    return np.clip(np.rint(out), 0, 255).astype(np.uint8)


def histogram(ch: np.ndarray) -> np.ndarray:
    return np.bincount(ch.ravel(), minlength=256)[:256]


def get_view(rgb: np.ndarray, mode: str, colorize: bool):
    """回傳 (顯示影像, 直方圖序列 [(label, counts, color)], 統計用 {label: channel})。"""
    if mode == "Original":
        gray = yuv_channel_u8(rgb, "Y")
        chans = {"R": rgb[..., 0], "G": rgb[..., 1], "B": rgb[..., 2], "Luma(Y)": gray}
        colors = {"R": "tab:red", "G": "tab:green", "B": "tab:blue", "Luma(Y)": "black"}
        series = [(k, histogram(v), colors[k]) for k, v in chans.items()]
        return rgb, series, chans

    if mode in ("R", "G", "B"):
        idx = "RGB".index(mode)
        ch = rgb[..., idx]
        if colorize:
            disp = np.zeros_like(rgb)
            disp[..., idx] = ch
        else:
            disp = ch
        color = {"R": "tab:red", "G": "tab:green", "B": "tab:blue"}[mode]
    else:
        ch = yuv_channel_u8(rgb, mode)
        disp = ch
        color = {"Y": "black", "U": "tab:purple", "V": "tab:orange"}[mode]
    return disp, [(mode, histogram(ch), color)], {mode: ch}


def plot_histograms(series, title: str, log: bool = False):
    fig, ax = plt.subplots(figsize=(7, 3.4))
    x = np.arange(256)
    for label, h, color in series:
        ax.fill_between(x, h, step="mid", alpha=0.25, color=color)
        ax.plot(x, h, color=color, lw=1.1, label=label)
    ax.set_xlim(0, 255)
    ax.set_xlabel("Pixel intensity (0-255)")
    ax.set_ylabel("Pixel count")
    ax.set_title(title)
    if log:
        ax.set_yscale("symlog", linthresh=1)
    ax.grid(alpha=0.3)
    if len(series) > 1:
        ax.legend(loc="upper right")
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# 3. 卷積核、濾波與 PSNR
# ----------------------------------------------------------------------------
def parse_kernel_text(text: str) -> np.ndarray:
    """文字 → 2D 陣列。列以換行或 ';' 分隔，元素以空白或 ',' 分隔，支援分數（如 1/16）。"""
    rows = [r.strip() for r in re.split(r"[\n;]+", text) if r.strip()]
    if not rows:
        raise ValueError("矩陣為空，請輸入數值。")
    mat = []
    for i, row in enumerate(rows, 1):
        vals = []
        for tok in (t for t in re.split(r"[,\s]+", row) if t):
            try:
                vals.append(float(Fraction(tok)))
            except (ValueError, ZeroDivisionError):
                raise ValueError(f"第 {i} 列含無法解析的內容：「{tok}」") from None
        mat.append(vals)
    if len({len(r) for r in mat}) != 1:
        raise ValueError("各列元素數量不一致，無法構成矩陣。")
    return np.array(mat, dtype=float)


def validate_kernel(k: np.ndarray) -> np.ndarray:
    if k.ndim != 2 or k.size == 0:
        raise ValueError("卷積核必須是二維矩陣。")
    r, c = k.shape
    if r != c:
        raise ValueError(f"非方陣（{r}×{c}）：卷積核必須為方陣，例如 3×3、5×5。")
    if r % 2 == 0:
        raise ValueError(f"尺寸 {r}×{r} 為偶數：請使用奇數邊長，才能有明確的中心點。")
    if r > MAX_KERNEL:
        raise ValueError(f"尺寸 {r}×{r} 過大，上限為 {MAX_KERNEL}×{MAX_KERNEL}。")
    if not np.all(np.isfinite(k)):
        raise ValueError("矩陣含空白、NaN 或無限值，請填入有限數值。")
    return k


def normalize_kernel(k: np.ndarray, mode: str, divisor: float = 1.0):
    """回傳 (有效 kernel, 警告訊息 or None)。總和為 0 或除數為 0 時防呆。"""
    if mode == "sum":
        s = float(k.sum())
        if abs(s) < 1e-12:
            return k, "kernel 總和為 0（常見於邊緣偵測/浮雕），無法以總和正規化，已略過正規化。"
        return k / s, None
    if mode == "custom":
        if divisor == 0:
            raise ValueError("自訂除數不可為 0。")
        return k / divisor, None
    return k, None


def convolve_cv(img: np.ndarray, k: np.ndarray, border_type: int) -> np.ndarray:
    """cv2.filter2D 為相關運算；先把 kernel 上下左右翻轉，即為嚴格的卷積。"""
    kf = np.ascontiguousarray(np.flip(k)).astype(np.float32)
    return cv2.filter2D(img.astype(np.float32), -1, kf, borderType=border_type)


def convolve_numpy(img: np.ndarray, k: np.ndarray, pad_mode: str) -> np.ndarray:
    """純 NumPy 卷積 (用於驗證)：out(y,x) = Σ K(i,j)·I(y-i, x-j)。"""
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    f = img.astype(np.float64)
    pads = [(ph, ph), (pw, pw)] + [(0, 0)] * (f.ndim - 2)
    p = np.pad(f, pads, mode=pad_mode)
    H, W = f.shape[:2]
    kf = np.flip(k)
    out = np.zeros_like(f)
    for i in range(kh):
        for j in range(kw):
            out += kf[i, j] * p[i : i + H, j : j + W]
    return out


def to_u8(x: np.ndarray) -> np.ndarray:
    return np.clip(np.rint(x), 0, 255).astype(np.uint8)


def mse(a: np.ndarray, b: np.ndarray) -> float:
    if a.shape != b.shape:
        raise ValueError("兩張影像尺寸不一致，無法計算 MSE/PSNR。")
    return float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))


def psnr(a: np.ndarray, b: np.ndarray, max_val: float = 255.0):
    """PSNR = 10·log10(MAX² / MSE)；MSE = 0 時回傳 inf。"""
    m = mse(a, b)
    if m == 0:
        return math.inf, 0.0
    return 10.0 * math.log10(max_val**2 / m), m


def fmt_psnr(v: float) -> str:
    return "∞ dB" if math.isinf(v) else f"{v:.2f} dB"


# ----------------------------------------------------------------------------
# 4. 人臉偵測
# ----------------------------------------------------------------------------
@st.cache_resource(show_spinner=False)
def get_haar():
    path = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
    cascade = cv2.CascadeClassifier(path)
    if cascade.empty():
        raise RuntimeError("無法載入 Haar Cascade 模型檔。")
    return cascade


def detect_haar(rgb, scale_factor=1.1, min_neighbors=5, min_size=30):
    gray = cv2.equalizeHist(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY))
    rects, _, weights = get_haar().detectMultiScale3(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=int(min_neighbors),
        minSize=(int(min_size), int(min_size)),
        outputRejectLevels=True,
    )
    if len(rects) == 0:
        return []
    weights = np.asarray(weights, dtype=float).reshape(-1)
    return [
        {"box": tuple(int(v) for v in r), "score": float(w)} for r, w in zip(rects, weights)
    ]


@st.cache_resource(show_spinner="首次使用：下載 YuNet 模型（約 230 KB）…")
def ensure_yunet() -> str:
    """依序尋找：./models/ → 暫存資料夾 → 線上下載。"""
    here = os.path.dirname(os.path.abspath(__file__))
    local = os.path.join(here, "models", YUNET_NAME)
    if os.path.isfile(local) and os.path.getsize(local) > 50_000:
        return local
    cached = os.path.join(tempfile.gettempdir(), YUNET_NAME)
    if os.path.isfile(cached) and os.path.getsize(cached) > 50_000:
        return cached
    last_err = None
    for url in YUNET_URLS:
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                blob = resp.read()
            if len(blob) < 50_000:  # 可能抓到 Git LFS 指標檔
                raise RuntimeError("下載內容過小，疑似非模型檔")
            with open(cached, "wb") as f:
                f.write(blob)
            return cached
        except Exception as e:  # noqa: BLE001
            last_err = e
    raise RuntimeError(f"YuNet 模型下載失敗：{last_err}")


def detect_yunet(rgb, model_path, score_th=0.7, nms_th=0.3):
    h, w = rgb.shape[:2]
    det = cv2.FaceDetectorYN.create(model_path, "", (w, h), float(score_th), float(nms_th), 5000)
    _, faces = det.detect(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    if faces is None:
        return []
    out = []
    for f in faces:
        x, y, bw, bh = (int(round(v)) for v in f[:4])
        x, y = max(0, x), max(0, y)
        bw, bh = min(bw, w - x), min(bh, h - y)
        out.append({"box": (x, y, bw, bh), "score": float(f[-1])})
    return out


def draw_boxes(rgb: np.ndarray, dets: list) -> np.ndarray:
    out = rgb.copy()
    h, w = out.shape[:2]
    thick = max(2, int(round(min(h, w) / 300)))
    fscale = max(0.5, min(h, w) / 800)
    for i, d in enumerate(dets, 1):
        x, y, bw, bh = d["box"]
        cv2.rectangle(out, (x, y), (x + bw, y + bh), (0, 255, 0), thick)
        label = f"#{i} {d['score']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fscale, thick)
        ty = y - 6 if y - 6 - th > 0 else y + bh + th + 6
        cv2.rectangle(out, (x, ty - th - 4), (x + tw + 4, ty + 4), (0, 0, 0), -1)
        cv2.putText(
            out, label, (x + 2, ty), cv2.FONT_HERSHEY_SIMPLEX, fscale, (0, 255, 0), thick, cv2.LINE_AA
        )
    return out


# ----------------------------------------------------------------------------
# UI：各分頁
# ----------------------------------------------------------------------------
def png_bytes(arr: np.ndarray) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return buf.getvalue()


def render_info(meta: dict):
    c = st.columns(5)
    c[0].metric("檔名", meta.get("name", "-"))
    c[1].metric("格式", meta["format"])
    w, h = meta["orig_size"]
    c[2].metric("解析度", f"{w} × {h}")
    c[3].metric("色彩維度", f"{meta['channels']} ch ({meta['mode']})")
    c[4].metric("檔案大小", human_size(meta["file_bytes"]))
    st.caption(
        f"位元深度：{meta['bit_depth']} bit／通道　|　處理用解析度：{meta['proc_size'][0]} × {meta['proc_size'][1]}"
        + ("（已依側邊欄設定縮小）" if meta["resized"] else "")
    )


def render_color_tab(rgb: np.ndarray):
    mode = st.radio(
        "檢視模式",
        ["Original", "R", "G", "B", "Y", "U", "V"],
        horizontal=True,
        help="R/G/B：RGB 分量；Y/U/V：BT.601 轉換。U、V 含負值，已依理論值域線性映射至 0–255。",
    )
    o1, o2 = st.columns(2)
    colorize = o1.checkbox("R/G/B 以對應顏色顯示", value=False, disabled=mode not in ("R", "G", "B"))
    log = o2.checkbox("直方圖縱軸使用對數刻度", value=False)

    disp, series, chans = get_view(rgb, mode, colorize)
    left, right = st.columns(2)
    with left:
        st.image(disp, caption=f"檢視：{mode}", width="stretch", clamp=True)
    with right:
        fig = plot_histograms(series, f"Histogram — {mode}", log)
        st.pyplot(fig)
        plt.close(fig)

    stats = pd.DataFrame(
        {
            "通道": list(chans.keys()),
            "平均": [round(float(v.mean()), 2) for v in chans.values()],
            "標準差": [round(float(v.std()), 2) for v in chans.values()],
            "最小": [int(v.min()) for v in chans.values()],
            "最大": [int(v.max()) for v in chans.values()],
        }
    )
    st.dataframe(stats, hide_index=True, width="stretch")
    with st.expander("YUV 轉換公式 (BT.601)"):
        st.latex(r"Y = 0.299R + 0.587G + 0.114B")
        st.latex(r"U = -0.14713R - 0.28886G + 0.436B")
        st.latex(r"V = 0.615R - 0.51499G - 0.10001B")
        st.caption(
            "U 的理論範圍 ±111.18、V 的理論範圍 ±156.83；顯示時以 (U+U_max)/(2·U_max)×255 線性映射。"
        )


def read_kernel_ui():
    """回傳 (kernel, error_msg)。"""
    preset = st.selectbox("預設核", list(PRESETS.keys()), index=2)
    if PRESETS[preset] is None:
        size = st.select_slider("自訂尺寸", options=[3, 5, 7, 9, 11], value=3)
        base = np.zeros((size, size))
        base[size // 2, size // 2] = 1.0
    else:
        base = PRESETS[preset]
        size = base.shape[0]

    mode = st.radio("輸入方式", ["表格編輯", "文字輸入"], horizontal=True)
    key = f"{preset}_{size}"
    try:
        if mode == "表格編輯":
            df = pd.DataFrame(base, columns=[f"c{j + 1}" for j in range(size)])
            edited = st.data_editor(
                df, key=f"ed_{key}", hide_index=True, num_rows="fixed", width="stretch"
            )
            k = edited.to_numpy(dtype=float, na_value=np.nan)
        else:
            default = "\n".join(" ".join(f"{v:g}" for v in row) for row in base)
            text = st.text_area(
                "每列一行，元素以空白或逗號分隔（可用分數，如 1/16）",
                value=default,
                height=150,
                key=f"tx_{key}",
            )
            k = parse_kernel_text(text)
        return validate_kernel(k), None
    except ValueError as e:
        return None, str(e)


def render_filter_tab(rgb: np.ndarray):
    left, right = st.columns([1, 2])
    with left:
        st.subheader("卷積核設定")
        kernel, err = read_kernel_ui()
        norm_label = st.radio("正規化", ["不正規化", "除以 kernel 總和", "自訂除數"], index=1)
        divisor = 1.0
        if norm_label == "自訂除數":
            divisor = st.number_input("除數", value=1.0, step=1.0, format="%.4f")
        bias = st.number_input("輸出偏移 (bias)", value=0.0, min_value=-255.0, max_value=255.0,
                               step=1.0, help="浮雕/邊緣核常用 +128 讓結果落在可視範圍。")
        target = st.radio("處理對象", ["RGB 三通道", "灰階"], horizontal=True)
        border_key = st.selectbox("邊界處理", list(BORDERS.keys()))
        verify = st.checkbox("以純 NumPy 實作交叉驗證（較慢）", value=False)

    with right:
        if err:
            st.error(f"❌ 卷積核輸入有誤：{err}")
            return
        try:
            k_eff, warn = normalize_kernel(
                kernel, {"不正規化": "none", "除以 kernel 總和": "sum", "自訂除數": "custom"}[norm_label], divisor
            )
        except ValueError as e:
            st.error(f"❌ {e}")
            return
        if warn:
            st.warning(warn)
        if not np.any(kernel):
            st.warning("kernel 全為 0，輸出將為全黑（加上 bias 則為常數）。")

        src = rgb if target == "RGB 三通道" else cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        cv_border, np_mode = BORDERS[border_key]
        raw = convolve_cv(src, k_eff, cv_border) + bias
        out = to_u8(raw)

        if verify:
            raw_np = convolve_numpy(src, k_eff, np_mode) + bias
            st.success(
                f"OpenCV 與純 NumPy 卷積之最大絕對差：{np.max(np.abs(raw - raw_np)):.2e}（浮點誤差範圍內即一致）"
            )

        # --- 顯示比對 ---
        disp_src = src if src.ndim == 3 else cv2.cvtColor(src, cv2.COLOR_GRAY2RGB)
        disp_out = out if out.ndim == 3 else cv2.cvtColor(out, cv2.COLOR_GRAY2RGB)
        view = st.radio("比對方式", ["並排", "疊加混合", "差異圖"], horizontal=True)
        if view == "並排":
            a, b = st.columns(2)
            a.image(disp_src, caption="濾波前 (Original)", width="stretch")
            b.image(disp_out, caption="濾波後 (Filtered)", width="stretch")
        elif view == "疊加混合":
            alpha = st.slider("濾波後影像比重", 0.0, 1.0, 0.5, 0.05)
            st.image(cv2.addWeighted(disp_src, 1 - alpha, disp_out, alpha, 0), width="stretch")
        else:
            gain = st.slider("差異放大倍率", 1, 10, 1)
            diff = np.abs(disp_src.astype(np.int16) - disp_out.astype(np.int16)) * gain
            st.image(np.clip(diff, 0, 255).astype(np.uint8), caption="|原圖 − 濾波後|", width="stretch")

        # --- PSNR ---
        val, m = psnr(src, out)
        m1, m2, m3 = st.columns(3)
        m1.metric("PSNR", fmt_psnr(val))
        m2.metric("MSE", f"{m:.3f}")
        m3.metric("Σ kernel（原 → 有效）", f"{kernel.sum():g} → {k_eff.sum():g}")
        if math.isinf(val):
            st.info("原圖與濾波後完全相同（MSE = 0），PSNR 為 ∞。")
        if out.ndim == 3:
            rows = []
            for i, name in enumerate("RGB"):
                v, mm = psnr(src[..., i], out[..., i])
                rows.append({"通道": name, "PSNR": fmt_psnr(v), "MSE": round(mm, 3)})
            st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")

        with st.expander("PSNR 公式與卷積定義"):
            st.latex(r"\mathrm{MSE}=\frac{1}{mn}\sum_{i=0}^{m-1}\sum_{j=0}^{n-1}\big[I(i,j)-K(i,j)\big]^2")
            st.latex(r"\mathrm{PSNR}=10\log_{10}\!\frac{\mathrm{MAX}^2}{\mathrm{MSE}}=20\log_{10}\!\frac{\mathrm{MAX}}{\sqrt{\mathrm{MSE}}},\quad \mathrm{MAX}=255")
            st.latex(r"(I*K)(y,x)=\sum_{i}\sum_{j}K(i,j)\,I(y-i,\,x-j)")
            st.caption("I 為原圖、K 為濾波後影像；RGB 時 MSE 對三通道所有像素取平均。")
            st.write("有效 kernel（正規化後）：")
            st.dataframe(pd.DataFrame(k_eff).round(4), hide_index=True, width="stretch")

        st.download_button("下載濾波結果 (PNG)", png_bytes(disp_out), "filtered.png", "image/png")


def render_face_tab(rgb: np.ndarray):
    method = st.radio(
        "偵測方法",
        ["YuNet（DNN，具信心度）", "Haar Cascade（經典特徵）"],
        horizontal=True,
    )
    try:
        if method.startswith("YuNet"):
            c1, c2 = st.columns(2)
            score_th = c1.slider("信心度門檻", 0.1, 0.99, 0.7, 0.01)
            nms_th = c2.slider("NMS 門檻", 0.1, 0.9, 0.3, 0.05)
            dets = detect_yunet(rgb, ensure_yunet(), score_th, nms_th)
            note = "信心度：YuNet 分類分數，範圍 0–1。"
        else:
            c1, c2, c3 = st.columns(3)
            scale = c1.slider("scaleFactor", 1.05, 1.5, 1.1, 0.01)
            neigh = c2.slider("minNeighbors", 1, 10, 5)
            min_size = c3.slider("最小人臉 (px)", 10, 200, 30)
            dets = detect_haar(rgb, scale, neigh, min_size)
            note = "信心度：Haar 的 levelWeight（cascade 最終階段權重），非 0–1 機率，數值越大越可信。"
    except Exception as e:  # noqa: BLE001
        st.error(f"人臉偵測失敗：{e}")
        if method.startswith("YuNet"):
            st.info(
                f"可改用 Haar Cascade，或手動下載 `{YUNET_NAME}` 放入專案的 `models/` 資料夾。"
            )
        return

    st.metric("偵測到的人臉數", len(dets))
    if not dets:
        st.warning("未偵測到人臉。可降低信心度門檻／minNeighbors，或換另一種方法。")
    left, right = st.columns([3, 2])
    left.image(draw_boxes(rgb, dets), caption="偵測結果", width="stretch")
    if dets:
        table = pd.DataFrame(
            [
                {"#": i, "x": d["box"][0], "y": d["box"][1], "w": d["box"][2], "h": d["box"][3],
                 "confidence": round(d["score"], 3)}
                for i, d in enumerate(dets, 1)
            ]
        )
        right.dataframe(table, hide_index=True, width="stretch")
    right.caption(note)


# ----------------------------------------------------------------------------
# 主程式
# ----------------------------------------------------------------------------
def main():
    st.set_page_config(page_title="Image Lab", page_icon="🖼️", layout="wide")
    st.title("🖼️ Image Lab — 影像處理期中專題")

    with st.sidebar:
        st.header("影像來源")
        up = st.file_uploader("上傳影像", type=ALLOWED_EXT)
        max_side = st.slider("最大處理邊長 (px)", 512, 4000, 1600, 128,
                             help="過大的影像會先縮小，避免雲端記憶體不足與運算過慢。")

    if up is None:
        st.info("👈 請先於側邊欄上傳 PNG / JPG / BMP 影像。")
        return

    try:
        rgb, meta = load_image(up.getvalue(), max_side)
    except Exception as e:  # noqa: BLE001
        st.error(f"無法讀取影像（檔案可能損毀或格式不符）：{e}")
        return
    meta = {**meta, "name": up.name}

    render_info(meta)
    t1, t2, t3 = st.tabs(["🎨 色彩空間與直方圖", "🧮 卷積濾波與 PSNR", "🙂 人臉偵測"])
    with t1:
        render_color_tab(rgb)
    with t2:
        render_filter_tab(rgb)
    with t3:
        render_face_tab(rgb)


if __name__ == "__main__":
    main()


Writing app.py


## 3. 下載人臉偵測模型

In [3]:
import os, urllib.request

os.makedirs("models", exist_ok=True)
dst = "models/face_detection_yunet_2023mar.onnx"
url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
try:
    if not (os.path.exists(dst) and os.path.getsize(dst) > 50_000):
        urllib.request.urlretrieve(url, dst)
    size = os.path.getsize(dst)
    print("YuNet 模型：", dst, f"{size:,} bytes", "✅" if size > 50_000 else "❌ 檔案過小（可能是 LFS 指標檔）")
except Exception as e:
    print("YuNet 下載失敗（App 內仍可改用 Haar Cascade）：", e)


YuNet 模型： models/face_detection_yunet_2023mar.onnx 232,589 bytes ✅


## 4. 啟動 Streamlit 並取得公開網址

In [4]:
import os, re, socket, subprocess, time, urllib.request
from IPython.display import HTML, display

PORT = 8501

# 若重複執行，先關閉舊的程序
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

# 1) 啟動 Streamlit
st_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        if urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2).read() == b"ok":
            print("✅ Streamlit 已啟動")
            break
    except Exception:
        time.sleep(1)
else:
    print(open("streamlit.log").read()[-2000:])
    raise RuntimeError("Streamlit 啟動失敗，請查看上方 log")

# 2) 取得 cloudflared（免註冊的臨時公開通道）
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

tun_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate",
     "--protocol", "http2"],   # http2 比預設的 QUIC 更不容易被網路環境擋住
    stdout=subprocess.DEVNULL, stderr=open("cloudflared.log", "w"),
)

# 3) 等通道「真的連上」(log 出現 Registered tunnel connection) 才印出網址
public_url = None
for _ in range(90):
    time.sleep(1)
    if tun_proc.poll() is not None:
        break  # cloudflared 已結束
    log = open("cloudflared.log").read()
    found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", log)
             if not u.startswith("https://api.")]
    if found and "Registered tunnel connection" in log:
        public_url = found[0]
        break

# 4) 再等網址的 DNS 可解析，避免太早點開出現 Error 1016
if public_url:
    host = public_url.split("//")[1]
    for _ in range(60):
        try:
            socket.gethostbyname(host)
            break
        except OSError:
            time.sleep(1)
    display(HTML(f"<h2>👉 <a href='{public_url}' target='_blank'>{public_url}</a></h2>"
                 "<p>點開網址即可使用 Image Lab。若看到 <b>Error 1016</b>，請等 30 秒後重新整理。"
                 "<br>⚠️ 重新執行本格會產生新網址，舊網址立刻失效。</p>"))
else:
    print(open("cloudflared.log").read()[-1500:])
    print("❌ 通道沒有連上，請重新執行此儲存格；仍失敗請把上方 log 貼給助教/我。")


✅ Streamlit 已啟動


## （選用）診斷：網址打不開時執行

In [5]:
import subprocess, urllib.request

def _alive(name):
    return subprocess.run(["pgrep", "-f", name], capture_output=True).returncode == 0

print("Streamlit 程序  :", "執行中 ✅" if _alive("streamlit run") else "已停止 ❌")
print("cloudflared 程序:", "執行中 ✅" if _alive("cloudflared tunnel") else "已停止 ❌")
try:
    ok = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=3).read()
    print("本機 8501 健康檢查:", ok.decode(), "✅")
except Exception as e:
    print("本機 8501 無法連線 ❌", e)
log = open("cloudflared.log").read()
print("通道已註冊      :", "是 ✅" if "Registered tunnel connection" in log else "否 ❌")
print("--- cloudflared.log 最後 800 字 ---")
print(log[-800:])


Streamlit 程序  : 執行中 ✅
cloudflared 程序: 執行中 ✅
本機 8501 健康檢查: ok ✅
通道已註冊      : 是 ✅
--- cloudflared.log 最後 800 字 ---
UIC connection successful" run_id=a355542d-351b-4a72-a4e9-0893a01fc657 status=pass target=region2.v2.argotunnel.com
2026-11-02T12:08:02Z INF precheck component="TCP Connectivity" details="HTTP/2 connection successful" run_id=a355542d-351b-4a72-a4e9-0893a01fc657 status=pass target=region1.v2.argotunnel.com
2026-11-02T12:08:02Z INF precheck component="TCP Connectivity" details="HTTP/2 connection successful" run_id=a355542d-351b-4a72-a4e9-0893a01fc657 status=pass target=region2.v2.argotunnel.com
2026-11-02T12:08:02Z INF precheck component="Cloudflare API" details="API is reachable" run_id=a355542d-351b-4a72-a4e9-0893a01fc657 status=pass target=api.cloudflare.com:443
2026-11-02T12:08:02Z INF precheck complete hard_fail=false run_id=a355542d-351b-4a72-a4e9-0893a01fc657 suggested_protocol=http2



## （選用）停止服務

In [ ]:
# 停止服務（用完可執行；關閉 Colab 執行階段也會一併停止）
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
print("已停止。")
